In [1]:
!pip install gradio reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 19.5 MB/s eta 0:00:00


In [2]:
import ast

class DocGenieAnalyzer:

    @staticmethod
    def extract_function_signature(code):
        tree = ast.parse(code)
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                return {
                    "name": node.name,
                    "params": [arg.arg for arg in node.args.args]
                }
        return None

In [4]:
code = """
def add(a, b):
    return a + b
"""

result = DocGenieAnalyzer.extract_function_signature(code)
print(result)

{'name': 'add', 'params': ['a', 'b']}


In [5]:
def generate_google_docstring(signature):
    if not signature:
        return "No function found."

    name = signature["name"]
    params = signature["params"]

    docstring = f'"""{name} function.\n\n'

    if params:
        docstring += "Args:\n"
        for param in params:
            docstring += f"    {param}: Description of {param}.\n"

    docstring += '\nReturns:\n'
    docstring += "    Result of the function.\n"
    docstring += '"""'

    return docstring

signature = DocGenieAnalyzer.extract_function_signature(code)
doc = generate_google_docstring(signature)
print(doc)

"""add function.

Args:
    a: Description of a.
    b: Description of b.

Returns:
    Result of the function.
"""


In [6]:
def analyze_function_logic(code):
    tree = ast.parse(code)

    has_loop = False
    has_condition = False
    has_math = False

    for node in ast.walk(tree):

        if isinstance(node, (ast.For, ast.While)):
            has_loop = True

        if isinstance(node, ast.If):
            has_condition = True

        if isinstance(node, (ast.Add, ast.Sub, ast.Mult, ast.Div)):
            has_math = True

    return {
        "has_loop": has_loop,
        "has_condition": has_condition,
        "has_math": has_math
    }

In [7]:
logic = analyze_function_logic(code)
print(logic)

{'has_loop': False, 'has_condition': False, 'has_math': True}


In [8]:
def generate_smart_docstring(signature, logic):
    if not signature:
        return "No function found."

    name = signature["name"]
    params = signature["params"]

    description = f"{name} function."

    if logic["has_loop"]:
        description += " This function uses a loop."
    if logic["has_condition"]:
        description += " It contains conditional statements."
    if logic["has_math"]:
        description += " It performs mathematical operations."

    docstring = f'"""{description}\n\n'

    if params:
        docstring += "Args:\n"
        for param in params:
            docstring += f"    {param}: Description of {param}.\n"

    docstring += '\nReturns:\n'
    docstring += "    Result of the function.\n"
    docstring += '"""'

    return docstring

In [9]:
signature = DocGenieAnalyzer.extract_function_signature(code)
logic = analyze_function_logic(code)

smart_doc = generate_smart_docstring(signature, logic)
print(smart_doc)

"""add function. It performs mathematical operations.

Args:
    a: Description of a.
    b: Description of b.

Returns:
    Result of the function.
"""


In [10]:
def generate_numpy_docstring(signature, logic):
    if not signature:
        return "No function found."

    name = signature["name"]
    params = signature["params"]

    description = f"{name} function."

    if logic["has_loop"]:
        description += " This function uses a loop."
    if logic["has_condition"]:
        description += " It contains conditional statements."
    if logic["has_math"]:
        description += " It performs mathematical operations."

    docstring = f'"""{description}\n\n'

    if params:
        docstring += "Parameters\n"
        docstring += "----------\n"
        for param in params:
            docstring += f"{param} : type\n"
            docstring += f"    Description of {param}.\n"

    docstring += "\nReturns\n"
    docstring += "-------\n"
    docstring += "type\n"
    docstring += "    Result of the function.\n"
    docstring += '"""'

    return docstring

In [12]:
signature = DocGenieAnalyzer.extract_function_signature(code)
logic = analyze_function_logic(code)

numpy_doc = generate_numpy_docstring(signature, logic)
print(numpy_doc)

"""add function. It performs mathematical operations.

Parameters
----------
a : type
    Description of a.
b : type
    Description of b.

Returns
-------
type
    Result of the function.
"""


In [15]:
import gradio as gr

def generate_doc(code, style):
    signature = DocGenieAnalyzer.extract_function_signature(code)
    logic = analyze_function_logic(code)

    if style == "Google":
        doc = generate_smart_docstring(signature, logic)
    else:
        doc = generate_numpy_docstring(signature, logic)

    return doc, create_pdf(doc)


interface = gr.Interface(
    fn=generate_doc,
    inputs=[
        gr.Textbox(lines=10, label="Enter Python Function Code"),
        gr.Radio(["Google", "NumPy"], value="Google", label="Select Docstring Style")
    ],
    outputs=[
        gr.Textbox(label="Generated Docstring"),
        gr.File(label="Download PDF")
    ],
    title="Doc-Genie - Python Docstring Generator"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4e019ea3ab66899f54.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter


def create_pdf(docstring_text):
    file_path = "generated_docstring.pdf"
    pdf = SimpleDocTemplate(file_path, pagesize=letter)
    elements = []

    styles = getSampleStyleSheet()
    style = styles["Normal"]

    paragraphs = docstring_text.split("\n")

    for line in paragraphs:
        elements.append(Paragraph(line.replace(" ", "&nbsp;"), style))
        elements.append(Spacer(1, 0.2 * inch))

    pdf.build(elements)

    return file_path